# ML Assignment 1 (Part 1)
# Group: P3-21
# Members: Evan Ting (2500985), Brian Yong (2502299), Nigel Ng (2500544), Troy Ang (2503451), Yap Kai Rei (2501155)

In [105]:
import numpy as np
import pandas as pd

In [106]:
# import all hdb datasets
df_2017 = pd.read_csv('./Resale flat prices based on registration date from Jan-2017 onwards.csv')
df_2015 = pd.read_csv('./ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv')
df_2012 = pd.read_csv('./ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv')
df_2000 = pd.read_csv('./ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv')
df_1990 = pd.read_csv('./ResaleFlatPricesBasedonApprovalDate19901999.csv')
df_1990.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price
0,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,9000
1,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,04 TO 06,31.0,IMPROVED,1977,6000
2,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,8000
3,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,07 TO 09,31.0,IMPROVED,1977,6000
4,1990-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,04 TO 06,73.0,NEW GENERATION,1976,47200


In [107]:
# concat vertically
df = pd.concat([df_1990, df_2000, df_2012, df_2015, df_2017], ignore_index=True)

# Check the result
print(f"Total rows: {len(df)}")

Total rows: 970092


In [108]:
# import postal code dataset
df_postal = pd.read_csv('sg_zipcode_mapper.csv', encoding='latin1')
df_postal = df_postal.rename(columns={'longtitude': 'longitude'})
df_postal.head()

,postal,latitude,longitude,searchval,blk_no,road_name,building,address,postal.1
0,398614,1.312763,103.883519,# 1 LOFT,1,LORONG 24 GEYLANG,# 1 LOFT,1 LORONG 24 GEYLANG # 1 LOFT SINGAPORE 398614,398614
1,398721,1.312390,103.881504,# 1 SUITES,1,LORONG 20 GEYLANG,# 1 SUITES,1 LORONG 20 GEYLANG # 1 SUITES SINGAPORE 398721,398721
2,629875,1.309135,103.679463,1 BENOI ROAD SINGAPORE 629875,1,BENOI ROAD,NIL,1 BENOI ROAD SINGAPORE 629875,629875
3,439731,1.305466,103.895674,1 BOSCOMBE ROAD SINGAPORE 439731,1,BOSCOMBE ROAD,NIL,1 BOSCOMBE ROAD SINGAPORE 439731,439731
4,659592,1.344619,103.749789,1 BUKIT BATOK STREET 22 SINGAPORE 659592,1,BUKIT BATOK STREET 22,NIL,1 BUKIT BATOK STREET 22 SINGAPORE 659592,659592


In [109]:
# METHOD 1: kaggle 2018 dataset

# standardise casing
df['street_name'] = df['street_name'].str.upper()
df_postal['road_name'] = df_postal['road_name'].str.upper()

# standarise naming convention
df_postal['road_name'] = df_postal['road_name'].str.replace('AVENUE', 'AVE', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('STREET', 'ST', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('ROAD', 'RD', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('LORONG', 'LOR', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('DRIVE', 'DR', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('JALAN', 'JLN', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('CRESCENT', 'CRES', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('CLOSE', 'CL', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('PLACE', 'PL', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('UPPER', 'UPP', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('TERRACE', 'TER', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('PARK', 'PK', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('HEIGHTS', 'HTS', regex=False)

df_postal['road_name'] = df_postal['road_name'].str.replace('CENTRAL', 'CTRL', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('CENTRE', 'CTR', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('NORTH', 'NTH', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('SOUTH', 'STH', regex=False)

df_postal['road_name'] = df_postal['road_name'].str.replace('BUKIT', 'BT', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('COMMONWEALTH', "C'WEALTH", regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('TANJONG', 'TG', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('GARDENS', 'GDNS', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('KAMPONG', 'KG', regex=False)
df_postal['road_name'] = df_postal['road_name'].str.replace('SAINT', 'ST.', regex=False)


# add postal, lat and long values to df using df_postal from kaggle 2018 dataset
df = df.merge(
    df_postal[['blk_no', 'road_name', 'postal', 'latitude', 'longitude']], 
    left_on=['block', 'street_name'], 
    right_on=['blk_no', 'road_name'], 
    how='left'
)

df = df.drop(columns=['blk_no', 'road_name'])

df.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,postal,latitude,longitude
0,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,9000.0,NaN,NaN,NaN,NaN
1,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,04 TO 06,31.0,IMPROVED,1977,6000.0,NaN,NaN,NaN,NaN
2,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,8000.0,NaN,NaN,NaN,NaN
3,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,07 TO 09,31.0,IMPROVED,1977,6000.0,NaN,NaN,NaN,NaN
4,1990-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,04 TO 06,73.0,NEW GENERATION,1976,47200.0,NaN,560216.0,1.366197,103.841505


In [110]:
# number of missing unique addresses left after method 1 (kaggle 2018 dataset)
df['full_address'] = df['block'].astype(str) + ' ' + df['street_name'].astype(str)
unique_addresses = df[df['postal'].isna()]['full_address'].unique()
len(unique_addresses)

745

In [111]:
# METHOD 2: onemap api

import requests
import time
from tqdm import tqdm

def get_location_details(address):
    """
    Fetches Postal Code, Latitude, and Longitude from OneMap API.
    """
    try:
        url = f"https://www.onemap.gov.sg/api/common/elastic/search?searchVal={address}&returnGeom=Y&getAddrDetails=Y"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if data['found'] > 0:
            result = data['results'][0]
            return {
                'postal': result.get('POSTAL'),
                'latitude': result.get('LATITUDE'),
                'longitude': result.get('LONGITUDE')
            }
    except Exception as e:
        return None
    return None

# unique list of addresses to search
df['full_address'] = df['block'].astype(str) + ' ' + df['street_name'].astype(str)
unique_addresses = df[df['postal'].isna()]['full_address'].unique()

print(f"Total unique addresses to process: {len(unique_addresses)}")

results_map = {}

for addr in tqdm(unique_addresses):
    details = get_location_details(addr)
    if details:
        results_map[addr] = details
    
    # prevent hitting rate limit
    time.sleep(0.5) 

for addr, details in results_map.items():
    mask = (df['full_address'] == addr) & (df['postal'].isna())
    
    df.loc[mask, 'postal'] = details['postal']
    df.loc[mask, 'latitude'] = details['latitude']
    df.loc[mask, 'longitude'] = details['longitude']

# clean up and export
df.drop(columns=['full_address'], inplace=True)

Total unique addresses to process: 745


100%|██████████| 745/745 [14:15<00:00,  1.15s/it]
C:\Users\USER\AppData\Local\Temp\ipykernel_27128\619476428.py:47: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '569953' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 'postal'] = details['postal']
C:\Users\USER\AppData\Local\Temp\ipykernel_27128\619476428.py:48: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.37701651576853' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 'latitude'] = details['latitude']
C:\Users\USER\AppData\Local\Temp\ipykernel_27128\619476428.py:49: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '103.834733409768' has dtype incompatible with float64, please expli

In [112]:
# found NIL value for postal (treat that as nan)
df.replace("NIL", np.nan, inplace=True)
# replace lat and long with null as long as postal is null
df['latitude'] = df['latitude'].mask(df['postal'].isna())
df['longitude'] = df['longitude'].mask(df['postal'].isna())

df[df[['postal','latitude','longitude']].isnull().any(axis=1)]

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,postal,latitude,longitude
15,1990-01,ANG MO KIO,3 ROOM,247,ANG MO KIO AVE 3,04 TO 06,82.0,NEW GENERATION,1978,56900.0,NaN,NaN,NaN,NaN
17,1990-01,ANG MO KIO,3 ROOM,252,ANG MO KIO AVE 4,07 TO 09,67.0,NEW GENERATION,1977,40000.0,NaN,NaN,NaN,NaN
65,1990-01,ANG MO KIO,3 ROOM,455,ANG MO KIO AVE 10,10 TO 12,67.0,NEW GENERATION,1980,47200.0,NaN,NaN,NaN,NaN
66,1990-01,ANG MO KIO,3 ROOM,455,ANG MO KIO AVE 10,10 TO 12,67.0,NEW GENERATION,1980,43500.0,NaN,NaN,NaN,NaN
353,1990-01,BUKIT BATOK,3 ROOM,4,HILLVIEW AVE,07 TO 09,82.0,NEW GENERATION,1978,50500.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
821125,2020-06,CHOA CHU KANG,4 ROOM,216,CHOA CHU KANG CTRL,01 TO 03,106.0,Model A,1990,355000.0,68 years 09 months,NaN,NaN,NaN
900010,2023-02,CHOA CHU KANG,4 ROOM,216,CHOA CHU KANG CTRL,04 TO 06,105.0,Model A,1990,475000.0,65 years 11 months,NaN,NaN,NaN
926603,2024-04,CHOA CHU KANG,4 ROOM,216,CHOA CHU KANG CTRL,04 TO 06,107.0,Model A,1990,530000.0,64 years 10 months,NaN,NaN,NaN
926607,2024-04,CHOA CHU KANG,4 ROOM,216,CHOA CHU KANG CTRL,01 TO 03,117.0,Model A,1990,535000.0,64 years 10 months,NaN,NaN,NaN


In [113]:
# number of missing unique addresses left after method 2 (onemap api)
df['full_address'] = df['block'].astype(str) + ' ' + df['street_name'].astype(str)
unique_addresses = df[df['postal'].isna()]['full_address'].unique()
len(unique_addresses)

119

In [ ]:
# METHOD 3: manual search

# manual query requires full naming convention
df['street_name'] = df['street_name'].str.replace('AVE', 'AVENUE', regex=False)
df['street_name'] = df['street_name'].str.replace('ST', 'STREET', regex=False)
df['street_name'] = df['street_name'].str.replace('RD', 'ROAD', regex=False)
df['street_name'] = df['street_name'].str.replace('LOR', 'LORONG', regex=False)
df['street_name'] = df['street_name'].str.replace('DR', 'DRIVE', regex=False)
df['street_name'] = df['street_name'].str.replace('JLN', 'JALAN', regex=False)
df['street_name'] = df['street_name'].str.replace('CRES', 'CRESCENT', regex=False)
df['street_name'] = df['street_name'].str.replace('CL', 'CLOSE', regex=False)
df['street_name'] = df['street_name'].str.replace('PL', 'PLACE', regex=False)
df['street_name'] = df['street_name'].str.replace('UPP', 'UPPER', regex=False)
df['street_name'] = df['street_name'].str.replace('TER', 'TERRACE', regex=False)
df['street_name'] = df['street_name'].str.replace('PK', 'PARK', regex=False)
df['street_name'] = df['street_name'].str.replace('HTS', 'HEIGHTS', regex=False)

df['street_name'] = df['street_name'].str.replace('CTRL', 'CENTRAL', regex=False)
df['street_name'] = df['street_name'].str.replace('CTR', 'CENTRE', regex=False)
df['street_name'] = df['street_name'].str.replace('NTH', 'NORTH', regex=False)
df['street_name'] = df['street_name'].str.replace('STH', 'SOUTH', regex=False)

df['street_name'] = df['street_name'].str.replace('BT', 'BUKIT', regex=False)
df['street_name'] = df['street_name'].str.replace("C'WEALTH", 'COMMONWEALTH', regex=False)
df['street_name'] = df['street_name'].str.replace('TG', 'TANJONG', regex=False)
df['street_name'] = df['street_name'].str.replace('GDNS', 'GARDENS', regex=False)
df['street_name'] = df['street_name'].str.replace('KG', 'KAMPONG', regex=False)
df['street_name'] = df['street_name'].str.replace('ST.', 'SAINT', regex=False)


mask = df[['postal', 'latitude', 'longitude']].isnull().any(axis=1)
unique_missing_df = df.loc[mask, ['block', 'street_name']].drop_duplicates()

# # export block + street_name to csv for manual query
# unique_missing_df.to_csv('manual_search_coords.csv', index=False)

# print(f"Exported {len(unique_missing_df)} unique addresses.")

In [115]:
df_manual_search = pd.read_csv('./manual_search_coords.csv')
df_manual_search

,block,street_name,postal,latitude,longitude
0,247,ANG MO KIO AVENUE 3,560247.0,1.3691,103.8437
1,252,ANG MO KIO AVENUE 4,560252.0,1.3737,103.8365
2,455,ANG MO KIO AVENUE 10,560455.0,1.3630,103.8564
3,4,HILLVIEW AVENUE,669591.0,1.3615,103.7614
4,6,HILLVIEW AVENUE,669592.0,1.3624,103.7618
...,...,...,...,...,...
114,24,KAMPONG BAHRU HILL,90024.0,1.2775,103.8330
115,68,DAKOTA CRESCENT,390068.0,1.3080,103.8885
116,12,REDHILL CLOSE,151012.0,1.2872,103.8172
117,216,CHOA CHU KANG CENTRAL,680216.0,1.3831,103.7469


In [116]:
# add postal, lat and long values to df using df_manual_search from manual search
df_merged = df.merge(
    df_manual_search[['block', 'street_name', 'postal', 'latitude', 'longitude']], 
    on=['block', 'street_name'], 
    how='left', 
    suffixes=('', '_manual')
)

df_merged['postal'] = df_merged['postal'].fillna(df_merged['postal_manual'])
df_merged['latitude'] = df_merged['latitude'].fillna(df_merged['latitude_manual'])
df_merged['longitude'] = df_merged['longitude'].fillna(df_merged['longitude_manual'])

df = df_merged.drop(columns=['postal_manual', 'latitude_manual', 'longitude_manual'])

In [117]:
# number of missing unique addresses left after method 3 (manual search)
df['full_address'] = df['block'].astype(str) + ' ' + df['street_name'].astype(str)
unique_addresses = df[df['postal'].isna()]['full_address'].unique()
len(unique_addresses)

11

In [118]:
# drop rows with null postal, lat or long
df = df.dropna(subset=['postal', 'latitude', 'longitude'])
# postal, lat and long have no more null values now
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 973275 entries, 0 to 974594
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   month                973275 non-null  object 
 1   town                 973275 non-null  object 
 2   flat_type            973275 non-null  object 
 3   block                973275 non-null  object 
 4   street_name          973275 non-null  object 
 5   storey_range         973275 non-null  object 
 6   floor_area_sqm       973275 non-null  float64
 7   flat_model           973275 non-null  object 
 8   lease_commence_date  973275 non-null  int64  
 9   resale_price         973275 non-null  float64
 10  remaining_lease      262376 non-null  object 
 11  postal               973275 non-null  object 
 12  latitude             973275 non-null  object 
 13  longitude            973275 non-null  object 
 14  full_address         973275 non-null  object 
dtypes: float64(2), int64(1

In [ ]:
# clean up and export whole df to csv
df = df.drop(columns=['full_address'])
df['postal'] = df['postal'].astype(int)
df['postal'] = df['postal'].astype(str)
df['latitude'] = df['latitude'].astype('float64')
df['longitude'] = df['longitude'].astype('float64')
print(df.info())
df.to_csv('hdb_resale_with_coords.csv', index=False)

print("Process Complete! Data exported to 'hdb_resale_with_coords.csv'")

Process Complete! Data exported to 'hdb_resale_with_coords.csv'
